# Customer Churn Forecasting - Data Exploration & Visualization
## Module 1: Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory analysis on the Telco Customer Churn dataset to understand customer demographics, service usage patterns, and churn distribution.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully.')

## 1.1 Load the Dataset

In [ ]:
# Load the Telco Customer Churn dataset
import os

if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found. Please place the CSV file in the project directory.')

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'\nColumn Names:\n{list(df.columns)}')

## 1.2 Initial Data Inspection

In [ ]:
# Display first few rows
print('=' * 80)
print('FIRST 5 ROWS')
print('=' * 80)
display(df.head())

print('\n' + '=' * 80)
print('LAST 5 ROWS')
print('=' * 80)
display(df.tail())

In [ ]:
# Dataset information
print('=' * 80)
print('DATASET INFO')
print('=' * 80)
print(df.info())

print('\n' + '=' * 80)
print('STATISTICAL SUMMARY')
print('=' * 80)
display(df.describe(include='all').T)

In [ ]:
# Check for missing values
print('=' * 80)
print('MISSING VALUES ANALYSIS')
print('=' * 80)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
else:
    print('No missing values found in the dataset.')

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicates}')

# Check data types
print('\nData Types:')
print(df.dtypes.value_counts())

## 1.3 Target Variable Analysis - Churn Distribution

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Churn Distribution:')
print(f'  No (Retained):  {churn_counts["No"]:,} ({churn_pct["No"]:.2f}%)')
print(f'  Yes (Churned):  {churn_counts["Yes"]:,} ({churn_pct["Yes"]:.2f}%)')
print(f'  Churn Rate:     {churn_pct["Yes"]:.2f}%')

# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2ecc71', '#e74c3c']
churn_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'], colors=colors,
            autopct='%1.1f%%', startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12, 'fontweight': 'bold'},
            shadow=True)
axes[1].set_title('Churn Rate Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nChurn distribution plot saved.')

## 1.4 Demographic Analysis

In [ ]:
# Gender distribution by churn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender vs Churn
gender_churn = pd.crosstab(df['gender'], df['Churn'], normalize='index') * 100
gender_churn.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Churn Rate by Gender', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(['Retained', 'Churned'])

# Senior Citizen vs Churn
senior_churn = pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index') * 100
senior_churn.index = ['Non-Senior', 'Senior']
senior_churn.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Churn Rate by Senior Citizen Status', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(['Retained', 'Churned'])

# Partner vs Churn
partner_churn = pd.crosstab(df['Partner'], df['Churn'], normalize='index') * 100
partner_churn.plot(kind='bar', ax=axes[2], color=colors, edgecolor='black', alpha=0.85)
axes[2].set_title('Churn Rate by Partner Status', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Percentage (%)')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.savefig('demographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Service Usage Analysis

In [ ]:
# Service features analysis
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=colors, edgecolor='black', alpha=0.85)
    axes[i].set_title(f'Churn by {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)
    axes[i].legend(['Retained', 'Churned'], fontsize=8)

plt.suptitle('Churn Rate by Service Subscriptions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('service_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.6 Numerical Features Distribution

In [ ]:
# Convert TotalCharges to numeric (handle spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Numerical features distribution by churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        subset = df[df['Churn'] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, label=f'Churn={label}', color=color, edgecolor='black')
    axes[i].set_title(f'{col} Distribution by Churn', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Box plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    sns.boxplot(x='Churn', y=col, data=df, ax=axes[i], palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
    axes[i].set_title(f'{col} by Churn Status', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('numerical_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.7 Contract and Payment Analysis

In [ ]:
# Contract type and Payment method analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Contract type
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(['Retained', 'Churned'])

# Payment method
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('Churn Rate by Payment Method', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)
axes[1].legend(['Retained', 'Churned'])

plt.tight_layout()
plt.savefig('contract_payment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.8 Correlation Analysis

In [ ]:
# Prepare numeric data for correlation
df_numeric = df.copy()
df_numeric['Churn_Binary'] = (df_numeric['Churn'] == 'Yes').astype(int)
df_numeric['TotalCharges'] = pd.to_numeric(df_numeric['TotalCharges'], errors='coerce')

# Select numeric columns for correlation
numeric_cols = df_numeric.select_dtypes(include=[np.number]).columns
corr_matrix = df_numeric[numeric_cols].corr()

# Correlation heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with Churn
churn_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(ascending=False)
print('\nCorrelation with Churn (Binary):')
print(churn_corr)

## 1.9 Key EDA Findings

### Summary of Insights:
1. **Churn Rate:** Approximately 26.5% of customers have churned - significant class imbalance exists
2. **Tenure Impact:** Customers with shorter tenure are much more likely to churn
3. **Contract Type:** Month-to-month contracts have the highest churn rate (~42%)
4. **Internet Service:** Fiber optic users show higher churn rates than DSL users
5. **Payment Method:** Electronic check users churn significantly more than other payment methods
6. **Senior Citizens:** Senior customers have nearly double the churn rate of non-seniors
7. **Monthly Charges:** Churned customers tend to have higher monthly charges
8. **Support Services:** Customers without online security, tech support, or backup services churn more